In [6]:
# Combines Census ZCTA population data with Psychology Today therapist counts.
# Outputs a clean CSV with Zip, Population, Count, Density, and Region label.
#
# Inputs:
#   - census_data/zcta_dhc_2020.csv         (population by ZCTA)
#   - therapist_counts/therapist_counts_by_zip.csv  (therapist counts)
#
# Output:
#   - combined_zip_population_therapists.csv

import csv
from pathlib import Path


# ---------- Configuration ----------
CENSUS_CSV     = Path("census_data/zcta_dhc_2020.csv")
THERAPIST_CSV  = Path("therapist_counts/therapist_counts_by_zip.csv")
OUTPUT_CSV     = Path("combined_zip_population_therapists.csv")


# ---------- Region label rubric ----------
# Each entry: (low_prefix_int, high_prefix_int, region_label)
# Prefixes are the first 3 digits of the ZIP, treated as integers.
# Ranges are inclusive on both ends.
REGION_RULES = [
    # Northern Virginia (NoVA)
    (201, 201, "Northern VA - North/West"),
    (220, 220, "Northern VA - Central NoVA"),
    (221, 221, "Northern VA - East/Central"),
    (222, 222, "Northern VA - Arlington"),
    (223, 223, "Northern VA - Alexandria"),
    # Central &amp; Eastern Virginia
    (230, 232, "Central/East VA - Richmond Area"),
    (233, 235, "Central/East VA - Hampton Roads/Tidewater"),
    (236, 237, "Central/East VA - Peninsula/Hampton Roads"),
    (238, 239, "Central/East VA - Southside/Central"),
    # Western &amp; Southwest Virginia
    (240, 241, "Western/SW VA - Roanoke Area"),
    (242, 242, "Western/SW VA - Far Southwest"),
    (243, 244, "Western/SW VA - Mountain Region"),
    (245, 245, "Western/SW VA - Southern/Piedmont"),
]


def label_zip(zip_code):
    """
    Returns the regional label for a 5-digit ZIP based on its 3-digit prefix.
    Returns 'Other' if no rule matches.
    """
    try:
        prefix = int(zip_code[:3])
    except (ValueError, TypeError):
        return "Other"

    for low, high, label in REGION_RULES:
        if low <= prefix <= high:
            return label
    return "Other"


# ---------- Load Census population data ----------
def load_population():
    """
    Reads census_data/zcta_dhc_2020.csv and returns a dict:
        { zip_string : population_int }
    
    The header is:
        NAME,H1_001N,zip code tabulation area
    Column 2 (H1_001N) becomes 'population'
    Column 3 ('zip code tabulation area') becomes 'zip'
    """
    pop = {}
    with open(str(CENSUS_CSV), "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)   # skip header line
        print("Census header: {}".format(header))

        skipped_blank = 0
        for row in reader:
            if len(row) < 3:
                continue
            
            # Column 3 is the ZIP — strip whitespace and zero-pad to 5 digits
            zip_raw = row[2].strip()
            if not zip_raw:
                skipped_blank += 1
                continue
            zip_code = zip_raw.zfill(5)

            # Column 2 is the population
            try:
                population = int(row[1].strip())
            except ValueError:
                population = 0

            pop[zip_code] = population

    print("Loaded population data for {} ZIPs ({} blank rows skipped)".format(
        len(pop), skipped_blank
    ))
    return pop


# ---------- Load therapist counts ----------
def load_therapist_counts():
    """
    Reads therapist_counts/therapist_counts_by_zip.csv and returns a dict:
        { zip_string : count_int }
    
    Skips any row that has a non-empty 'error' column.
    """
    counts = {}
    skipped_error = 0
    skipped_blank = 0

    with open(str(THERAPIST_CSV), "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            zip_raw = (row.get("zip") or "").strip()
            if not zip_raw:
                skipped_blank += 1
                continue
            zip_code = zip_raw.zfill(5)

            # Skip if there's an error message
            error = (row.get("error") or "").strip()
            if error:
                skipped_error += 1
                continue

            # Parse the count
            count_raw = (row.get("result_count") or "").strip()
            if not count_raw:
                skipped_blank += 1
                continue
            try:
                count = int(count_raw)
            except ValueError:
                skipped_blank += 1
                continue

            counts[zip_code] = count

    print("Loaded therapist counts for {} ZIPs".format(len(counts)))
    print("  Skipped due to error  : {}".format(skipped_error))
    print("  Skipped blank/invalid : {}".format(skipped_blank))
    return counts


# ---------- Combine and write output ----------
def combine_and_write(population, counts):
    """
    Joins the two dicts on ZIP (exact match), filters out zero-population
    rows, computes Density, adds Region label, and writes the output CSV.
    """
    rows_out = []
    matched = 0
    skipped_no_pop_match = 0
    skipped_no_count_match = 0
    skipped_zero_pop = 0

    # Iterate over every ZIP that has a therapist count (these are the
    # ones we requested). Match against the population dict.
    all_zips = set(counts.keys()) | set(population.keys())

    for zip_code in sorted(all_zips):
        count = counts.get(zip_code)
        pop = population.get(zip_code)

        # Must exist in BOTH files
        if count is None:
            skipped_no_count_match += 1
            continue
        if pop is None:
            skipped_no_pop_match += 1
            continue

        # Drop zero-population ZIPs
        if pop == 0:
            skipped_zero_pop += 1
            continue

        # Compute density: therapists per 10,000 residents
        density = (count / pop) * 10000.0

        # Regional label
        region = label_zip(zip_code)

        rows_out.append({
            "Zip":        zip_code,
            "Population": pop,
            "Count":      count,
            "Density":    round(density, 4),
            "Region":     region,
        })
        matched += 1

    # Write CSV
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = ["Zip", "Population", "Count", "Density", "Region"]
    with open(str(OUTPUT_CSV), "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows_out)

    print("\n--- Join Summary ---")
    print("Matched (in output)         : {}".format(matched))
    print("Therapist ZIPs not in Census: {}".format(skipped_no_pop_match))
    print("Census ZIPs not in counts   : {}".format(skipped_no_count_match))
    print("Dropped zero-population     : {}".format(skipped_zero_pop))
    print("\nWrote {} rows to {}".format(matched, OUTPUT_CSV))


# ---------- Optional: print quick stats by region ----------
def print_region_summary():
    """
    Reads back the output CSV and prints counts/density by region.
    """
    if not OUTPUT_CSV.exists():
        return

    by_region = {}
    with open(str(OUTPUT_CSV), "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            r = row["Region"]
            by_region.setdefault(r, {"zips": 0, "pop": 0, "count": 0})
            by_region[r]["zips"]  += 1
            by_region[r]["pop"]   += int(row["Population"])
            by_region[r]["count"] += int(row["Count"])

    print("\n--- Summary by Region ---")
    print("{:50s} {:6s} {:12s} {:10s} {:10s}".format(
        "Region", "ZIPs", "Population", "Therapists", "Density"
    ))
    print("-" * 92)
    for region in sorted(by_region.keys()):
        d = by_region[region]
        density = (d["count"] / d["pop"] * 10000.0) if d["pop"] else 0
        print("{:50s} {:6d} {:12,d} {:10,d} {:10.2f}".format(
            region, d["zips"], d["pop"], d["count"], density
        ))


# ---------- Entry point ----------
if __name__ == "__main__":
    print("Loading Census population data...")
    population = load_population()

    print("\nLoading therapist counts...")
    counts = load_therapist_counts()

    print("\nJoining datasets...")
    combine_and_write(population, counts)

    print_region_summary()

Loading Census population data...
Census header: ['NAME', 'H1_001N', 'zip code tabulation area']
Loaded population data for 33774 ZIPs (0 blank rows skipped)

Loading therapist counts...
Loaded therapist counts for 337 ZIPs
  Skipped due to error  : 221
  Skipped blank/invalid : 0

Joining datasets...

--- Join Summary ---
Matched (in output)         : 199
Therapist ZIPs not in Census: 136
Census ZIPs not in counts   : 33573
Dropped zero-population     : 2

Wrote 199 rows to combined_zip_population_therapists.csv

--- Summary by Region ---
Region                                             ZIPs   Population   Therapists Density   
--------------------------------------------------------------------------------------------
Northern VA - Alexandria                               14      155,066      2,262     145.87
Northern VA - Arlington                                10      119,505      2,066     172.88
Northern VA - Central NoVA                             19      185,189      2,857 